In [1]:
#Loading Packages
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
#Loading the Data
df_cohort = pd.read_csv('../data/cohort/cohort_analysis.csv',
                        parse_dates = ["acquisition_date", 'cancellation_month'])

df_cohort.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard
3,4,2024-07-01,NaT,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic


In [3]:
df_cohort["acquisition_date"] = pd.to_datetime(
    df_cohort["acquisition_date"],
    errors="coerce"
)

df_cohort["cancellation_month"] = pd.to_datetime(
    df_cohort["cancellation_month"],
    errors="coerce")

In [4]:
# Cohort (monthly)
df_cohort["acquisition_month"] = df_cohort["acquisition_date"].dt.to_period("M").dt.to_timestamp()

In [5]:
# New customers per month
cohort_size = (
    df_cohort.groupby("acquisition_month")
      .agg(new_users=("user_id", "count"))
      .reset_index()
)

In [6]:
#Calculate tenure (month difference)
#If not cancelled → treat as still active (NaT stays)

df_cohort["month_churned"] = (
    (df_cohort["cancellation_month"].dt.year - df_cohort["acquisition_month"].dt.year) * 12 
    +
    (df_cohort["cancellation_month"].dt.month - df_cohort["acquisition_month"].dt.month)
)

#Remove negative or invalid
df_cohort.loc[df_cohort["month_churned"] < 0, "month_churned"] = np.nan

In [7]:
#Count churn users by cohort & month
cohort_data = (
    df_cohort.dropna(subset=["month_churned"])
      .groupby(["acquisition_month", "month_churned"])
      .agg(users=("user_id", "count"))
      .reset_index()
)

cohort_data["month_churned"] = cohort_data["month_churned"].astype(int)

In [8]:
#Create full 12-month grid (critical)
max_month = 12
tenure_range = np.arange(0, max_month + 1)

full_index = pd.MultiIndex.from_product(
    [
        cohort_data["acquisition_month"].unique(),
        tenure_range
    ],
    names=["acquisition_month", "month_churned"]
)

cohort_data = (
    cohort_data
    .set_index(["acquisition_month", "month_churned"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

In [9]:
#merge cohort size
cohort_data = cohort_data.merge(cohort_size, on="acquisition_month", how="left")

In [10]:
#camulative churn
cohort_data = cohort_data.sort_values(["acquisition_month", "month_churned"])

cohort_data["cumulative_churn"] = (
    cohort_data
    .groupby("acquisition_month")["users"]
    .cumsum()
)

In [11]:
#Metrics
cohort_data["churn_rate"] = cohort_data["cumulative_churn"] / cohort_data["new_users"]
cohort_data["retention_rate"] = 1 - cohort_data["churn_rate"]

In [20]:
#Pivot for Heatmap
heatmap_data = cohort_data.pivot(
    index="acquisition_month",
    columns="month_churned",
    values="retention_rate"
)

# Optional: sort cohorts
heatmap_data = heatmap_data.sort_index()

In [21]:
heatmap_data = heatmap_data.loc[:, 1:]

In [25]:
for i in range(len(heatmap_data)):
    heatmap_data.iloc[i, (len(heatmap_data.columns) - i):] = np.nan

In [ ]:
fig = px.imshow(
    heatmap_data,
    text_auto=".0%",
    aspect="auto",
    color_continuous_scale=[
        "#d73027", "#f46d43", "#fdae61",
        "#fee08b", "#d9ef8b", "#66bd63", "#1a9850"
    ],
    title="Cohort Retention Heatmap (Monthly)"
)

In [23]:
fig.update_layout(
    xaxis_title="Months Since Acquisition",
    yaxis_title="Cohort"
)